In [1]:
import pandas as pd
import numpy as np
import utils.weatherMeth as wm

In [37]:
# Daily Long Period Average (LPA) of the rainfall derivative by NCDEX (30-year reference average from 1991 to 2020)
LPA_START_YEAR = 1991
LPA_END_YEAR = 2020

MONSOON_START_DATE = "06-01" # June 1 as per IMD and NCDEX specifications
MONSOON_END_DATE = "09-30"   # September 30 as per IMD and NCDEX specifications

CURRENT_YEAR = 2026

MUMBAI_MONSOON_ANCHOR_POINT = 2206.7 # As per IMD Mumbai Monsoon Season LPA in mm

In [9]:
CURRENT_DAILY_JUNE = pd.DataFrame({
    'date': pd.date_range(start='2024-06-01', periods=30, freq='D'),
    'IMD_Colaba': np.round(np.linspace(0.00, 25.00, 30), 2),
    'IMD_SantaCruz': np.round(np.linspace(0.50, 25.00, 30), 2)
})
CURRENT_DAILY_JUNE.head()

,date,IMD_Colaba,IMD_SantaCruz
0,2024-06-01,0.00,0.50
1,2024-06-02,0.86,1.34
2,2024-06-03,1.72,2.19
3,2024-06-04,2.59,3.03
4,2024-06-05,3.45,3.88


In [12]:
rng = np.random.default_rng(2026)
years = range(LPA_START_YEAR, LPA_END_YEAR + 1)
dates = pd.to_datetime([f"{year}-06-{day:02d}" for year in years for day in range(1, 31)])

LPA_DAILY_JUNE = pd.DataFrame({
    'date': dates,
    'IMD_Colaba': np.round(rng.uniform(0.00, 50.00, size=len(dates)), 2),
    'IMD_SantaCruz': np.round(rng.uniform(0.00, 50.00, size=len(dates)), 2),
})
LPA_DAILY_JUNE

,date,IMD_Colaba,IMD_SantaCruz
0,1991-06-01,8.95,31.64
1,1991-06-02,32.00,47.75
2,1991-06-03,23.36,20.33
3,1991-06-04,18.53,18.46
4,1991-06-05,17.75,4.29
...,...,...,...
895,2020-06-26,39.80,39.19
896,2020-06-27,49.57,44.87
897,2020-06-28,34.36,29.67
898,2020-06-29,32.60,12.52


In [13]:
def calculate_daily_city_rainfall_from_stations(city:str, dataset, s1:str, s2:str, total_stations:int):
    
    city_col = "IMD_"+city

    # NCDEX defined formula for city rainfall calc. (average of the stations)
    dataset[city_col] = (dataset[s1]+dataset[s2])/total_stations

    return dataset

### update the function ###
# city should calculate the total no. of stations based on the no. of columns that are stations.
 # stations start with "IMD_" and then area name. So we can filter columns based on that.

In [14]:
LPA_DAILY_JUNE = calculate_daily_city_rainfall_from_stations("Mumbai", LPA_DAILY_JUNE, s1="IMD_Colaba", s2="IMD_SantaCruz", total_stations=2)

In [16]:
CURRENT_DAILY_JUNE = calculate_daily_city_rainfall_from_stations("Mumbai", CURRENT_DAILY_JUNE, s1="IMD_Colaba", s2="IMD_SantaCruz", total_stations=2)

In [20]:
mumbai_daily_rainfall = CURRENT_DAILY_JUNE["IMD_Mumbai"]

In [28]:
# daily LPA for Mumbai: average of station means for each day (June 1-30) across 1991-2020
lpa_colaba = HISTORICAL_DAILY_JUNE.groupby(HISTORICAL_DAILY_JUNE['date'].dt.day)['IMD_Colaba'].mean().sort_index()
lpa_santacruz = HISTORICAL_DAILY_JUNE.groupby(HISTORICAL_DAILY_JUNE['date'].dt.day)['IMD_SantaCruz'].mean().sort_index()
mumbai_daily_LPA = ((lpa_colaba + lpa_santacruz) / 2).reset_index(drop=True)
mumbai_daily_LPA.name = 'IMD_Mumbai'

In [29]:
mumbai_daily_deviation = mumbai_daily_rainfall - mumbai_daily_LPA

In [38]:
RAINMUMBAI_MONTHLY_SPOT = pd.DataFrame({
    'date': pd.date_range(start=f"{CURRENT_YEAR}-06-01", periods=30, freq="D").strftime("%m-%d"),
    'Mumbai Daily LPA (in mm)': mumbai_daily_LPA.values,
    'Mumbai Daily Rainfall (in mm)': mumbai_daily_rainfall.values,
    'Mumbai Daily Deviation (mm)': mumbai_daily_deviation.values
})
RAINMUMBAI_MONTHLY_SPOT

,date,Mumbai Daily LPA (in mm),Mumbai Daily Rainfall (in mm),Mumbai Daily Deviation (mm)
0,06-01,24.348333,0.250,-24.098333
1,06-02,30.347167,1.100,-29.247167
2,06-03,27.223833,1.955,-25.268833
3,06-04,23.996167,2.810,-21.186167
4,06-05,24.055333,3.665,-20.390333
5,06-06,26.236667,4.515,-21.721667
6,06-07,26.266667,5.370,-20.896667
7,06-08,25.749500,6.220,-19.529500
8,06-09,27.669833,7.080,-20.589833
9,06-10,22.407000,7.930,-14.477000


In [39]:
RAINMUMBAI_MONTHLY_SPOT["Deviation Direction"] = np.where(RAINMUMBAI_MONTHLY_SPOT["Mumbai Daily Deviation (mm)"] < 0, "Negative", "Positive")

In [ ]:
RAINMUMBAI_MONTHLY_SPOT["SPOT of the Day"] = MUMBAI_MONSOON_ANCHOR_POINT + RAINMUMBAI_MONTHLY_SPOT["Mumbai Daily Deviation (mm)"].cumsum()

,date,Mumbai Daily LPA (in mm),Mumbai Daily Rainfall (in mm),Mumbai Daily Deviation (mm),Deviation Direction
0,06-01,24.348333,0.250,-24.098333,Negative
1,06-02,30.347167,1.100,-29.247167,Negative
2,06-03,27.223833,1.955,-25.268833,Negative
3,06-04,23.996167,2.810,-21.186167,Negative
4,06-05,24.055333,3.665,-20.390333,Negative
5,06-06,26.236667,4.515,-21.721667,Negative
6,06-07,26.266667,5.370,-20.896667,Negative
7,06-08,25.749500,6.220,-19.529500,Negative
8,06-09,27.669833,7.080,-20.589833,Negative
9,06-10,22.407000,7.930,-14.477000,Negative


In [42]:
print(f"Monthly LPA Value: {RAINMUMBAI_MONTHLY_SPOT['Mumbai Daily LPA (in mm)'].sum()}")
print(f"Monthly Rainfall Value: {RAINMUMBAI_MONTHLY_SPOT['Mumbai Daily Rainfall (in mm)'].sum()}")

Monthly LPA Value: 764.2728333333334
Monthly Rainfall Value: 378.74999999999994
